# 01 - Extract LOCA2 6 km (CMIP6, Scripps)

Daily `pr`, `tasmax`, `tasmin` for the configured GCM ensemble, scenarios SSP2-4.5 and
SSP3-7.0 plus historical 1950-2014, subset to the Tahoe bbox.

**Access strategy** (verified 2026-07-29): region-split NetCDF files on
`cirrus.ucsd.edu/~pierce/LOCA2/CONUS_regions_split/` are ~306 MB each with HTTP byte-range
support. We open them lazily over HTTP (`fsspec` + `h5netcdf`) and pull only the bbox
window - tens of MB per file land on disk, never the full region file. Requires
`h5netcdf` (see `environment.md`); the discovery/plan section runs without it.

In [1]:
import sys
print("Python:", sys.executable)

import warnings
from pathlib import Path
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import xarray as xr

# Pipeline root = climate/ (parent of notebooks/)
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from src.io import load_config, get_logger, append_manifest, sha256_file

cfg = load_config()
log = get_logger("01_extract_loca2")

RAW = ROOT / cfg["paths"]["raw"]
PROCESSED = ROOT / cfg["paths"]["processed"]
OUTPUTS = ROOT / cfg["paths"]["outputs"]
for p in (RAW, PROCESSED, OUTPUTS):
    p.mkdir(parents=True, exist_ok=True)

BBOX = cfg["study_area"]["bbox"]
log.info(f"bbox: lon {BBOX['lon_min']}..{BBOX['lon_max']}, lat {BBOX['lat_min']}..{BBOX['lat_max']}")

Python: C:\Program Files\ArcGIS\Pro\bin\Python\envs\arcgispro-py3\python.exe


2026-07-29 16:44:37 | INFO | 01_extract_loca2 | Log file: C:\Users\mbindl\Documents\GitHub\PROTECT\climate\logs\01_extract_loca2_2026-07-29_164437.log


2026-07-29 16:44:37 | INFO | 01_extract_loca2 | bbox: lon -120.5..-119.5, lat 38.5..39.5


## Discover files
List the actual per-GCM directories (the daily file is the one without a
`.monthly`/`.yearly` suffix) and build the download plan with HEAD sizes.

In [2]:
import re
import requests
import urllib3
urllib3.disable_warnings()  # cirrus has an incomplete cert chain; content is public data

L = cfg["sources"]["loca2"]
SESSION = requests.Session()
SESSION.verify = False


def ls_dir(url):
    r = SESSION.get(url, timeout=60)
    r.raise_for_status()
    return [h for h in re.findall(r'href="([^"]+)"', r.text) if not h.startswith(("?", "/"))]


def daily_files(gcm, scenario, var, region):
    member = L.get("member_overrides", {}).get(gcm, L["member"])
    url = (f"{L['base_url']}/{gcm}/{region}/{L['grid_dir']}/{member}/{scenario}/{var}/")
    try:
        names = ls_dir(url)
    except Exception as e:
        log.warning(f"listing failed: {url} ({e})")
        return []
    daily = [n for n in names if n.endswith(".nc")
             and ".monthly" not in n and ".yearly" not in n]
    return [url + n for n in daily]


plan = []
for gcm in L["gcms"]:
    for scen in ["historical"] + L["scenarios"]:
        for var in L["variables"]:
            for url in daily_files(gcm, scen, var, L["region"]):
                plan.append({"gcm": gcm, "scenario": scen, "variable": var, "url": url,
                             "filename": url.rsplit("/", 1)[1]})
plan = pd.DataFrame(plan)
log.info(f"{len(plan)} daily files across {plan['gcm'].nunique() if len(plan) else 0} GCMs")

# HEAD sizes (server supports ranges - we will not download these in full)
sizes = []
for url in plan["url"]:
    try:
        h = SESSION.head(url, timeout=60)
        sizes.append(int(h.headers.get("Content-Length", 0)))
    except Exception:
        sizes.append(None)
plan["remote_mb"] = pd.Series(sizes, dtype="float64") / 1e6
plan.to_csv(OUTPUTS / "loca2_plan.csv", index=False)
log.info(f"plan: {plan['remote_mb'].sum():,.0f} MB remote total (bbox subsets will be far smaller)")
plan.head(12)

2026-07-29 16:44:56 | WARNING | 01_extract_loca2 | listing failed: https://cirrus.ucsd.edu/~pierce/LOCA2/CONUS_regions_split/MPI-ESM1-2-HR/west/0p0625deg/r3i1p1f1/ssp245/pr/ (404 Client Error: Not Found for url: https://cirrus.ucsd.edu/~pierce/LOCA2/CONUS_regions_split/MPI-ESM1-2-HR/west/0p0625deg/r3i1p1f1/ssp245/pr/)


2026-07-29 16:44:56 | WARNING | 01_extract_loca2 | listing failed: https://cirrus.ucsd.edu/~pierce/LOCA2/CONUS_regions_split/MPI-ESM1-2-HR/west/0p0625deg/r3i1p1f1/ssp245/tasmax/ (404 Client Error: Not Found for url: https://cirrus.ucsd.edu/~pierce/LOCA2/CONUS_regions_split/MPI-ESM1-2-HR/west/0p0625deg/r3i1p1f1/ssp245/tasmax/)


2026-07-29 16:44:56 | WARNING | 01_extract_loca2 | listing failed: https://cirrus.ucsd.edu/~pierce/LOCA2/CONUS_regions_split/MPI-ESM1-2-HR/west/0p0625deg/r3i1p1f1/ssp245/tasmin/ (404 Client Error: Not Found for url: https://cirrus.ucsd.edu/~pierce/LOCA2/CONUS_regions_split/MPI-ESM1-2-HR/west/0p0625deg/r3i1p1f1/ssp245/tasmin/)


2026-07-29 16:44:56 | INFO | 01_extract_loca2 | 96 daily files across 5 GCMs


2026-07-29 16:45:02 | INFO | 01_extract_loca2 | plan: 56,182 MB remote total (bbox subsets will be far smaller)


,gcm,scenario,variable,url,filename,remote_mb
0,ACCESS-CM2,historical,pr,https://cirrus.ucsd.edu/~pierce/LOCA2/CONUS_re...,pr.ACCESS-CM2.historical.r1i1p1f1.1950-2014.LO...,306.492422
1,ACCESS-CM2,historical,tasmax,https://cirrus.ucsd.edu/~pierce/LOCA2/CONUS_re...,tasmax.ACCESS-CM2.historical.r1i1p1f1.1950-201...,1508.278561
2,ACCESS-CM2,historical,tasmin,https://cirrus.ucsd.edu/~pierce/LOCA2/CONUS_re...,tasmin.ACCESS-CM2.historical.r1i1p1f1.1950-201...,1515.629266
3,ACCESS-CM2,ssp245,pr,https://cirrus.ucsd.edu/~pierce/LOCA2/CONUS_re...,pr.ACCESS-CM2.ssp245.r1i1p1f1.2015-2044.LOCA_1...,141.483175
4,ACCESS-CM2,ssp245,pr,https://cirrus.ucsd.edu/~pierce/LOCA2/CONUS_re...,pr.ACCESS-CM2.ssp245.r1i1p1f1.2045-2074.LOCA_1...,137.757753
5,ACCESS-CM2,ssp245,pr,https://cirrus.ucsd.edu/~pierce/LOCA2/CONUS_re...,pr.ACCESS-CM2.ssp245.r1i1p1f1.2075-2100.LOCA_1...,123.252525
6,ACCESS-CM2,ssp245,tasmax,https://cirrus.ucsd.edu/~pierce/LOCA2/CONUS_re...,tasmax.ACCESS-CM2.ssp245.r1i1p1f1.2015-2044.LO...,696.229748
7,ACCESS-CM2,ssp245,tasmax,https://cirrus.ucsd.edu/~pierce/LOCA2/CONUS_re...,tasmax.ACCESS-CM2.ssp245.r1i1p1f1.2045-2074.LO...,696.225797
8,ACCESS-CM2,ssp245,tasmax,https://cirrus.ucsd.edu/~pierce/LOCA2/CONUS_re...,tasmax.ACCESS-CM2.ssp245.r1i1p1f1.2075-2100.LO...,603.385633
9,ACCESS-CM2,ssp245,tasmin,https://cirrus.ucsd.edu/~pierce/LOCA2/CONUS_re...,tasmin.ACCESS-CM2.ssp245.r1i1p1f1.2015-2044.LO...,699.444183


## Subset and save
Lazy-open each file over HTTP, slice the bbox, load only that window, and write
`data/raw/loca2/<original-name>__tahoe.nc`. Skips files already present unless
`run.overwrite_downloads`. LOCA2 longitudes are 0-360; the slice handles both.

In [3]:
try:
    import h5netcdf  # noqa: F401
    import fsspec
    HAS_H5 = True
except ImportError:
    HAS_H5 = False
    log.warning("h5netcdf not installed - skipping subsets. "
                "conda install -n arcgispro-py3 -c conda-forge h5netcdf")


def bbox_slice(ds):
    lon_min, lon_max = BBOX["lon_min"], BBOX["lon_max"]
    if float(ds.lon.max()) > 180:          # 0-360 convention
        lon_min, lon_max = lon_min + 360, lon_max + 360
    lat_asc = bool(ds.lat[0] < ds.lat[-1])
    lat_sl = slice(BBOX["lat_min"], BBOX["lat_max"]) if lat_asc else \
             slice(BBOX["lat_max"], BBOX["lat_min"])
    return ds.sel(lon=slice(lon_min, lon_max), lat=lat_sl)


loca2_dir = RAW / "loca2"
loca2_dir.mkdir(exist_ok=True)
done = skipped = failed = 0

if HAS_H5 and len(plan):
    for _, row in plan.iterrows():
        out = loca2_dir / (row["filename"].replace(".nc", "") + "__tahoe.nc")
        if out.exists() and not cfg["run"]["overwrite_downloads"]:
            skipped += 1
            continue
        try:
            with fsspec.open(row["url"], "rb", ssl=False) as f:
                ds = xr.open_dataset(f, engine="h5netcdf")
                sub = bbox_slice(ds).load()
            if min(sub.sizes.get("lat", 0), sub.sizes.get("lon", 0)) == 0:
                log.warning(f"empty bbox slice in {row['filename']} - "
                            f"check region_fallback ({L['region_fallback']})")
                failed += 1
                continue
            sub.to_netcdf(out, encoding={v: {"zlib": True, "complevel": 4}
                                         for v in sub.data_vars})
            append_manifest({"file": str(out.relative_to(ROOT)), "source_url": row["url"],
                             "size_bytes": out.stat().st_size, "sha256": sha256_file(out),
                             "retrieved_date": str(pd.Timestamp.today().date()),
                             "notebook": "01_extract_loca2"})
            done += 1
            log.info(f"[{done}] {out.name}  ({out.stat().st_size/1e6:.1f} MB)")
        except Exception as e:
            failed += 1
            log.warning(f"FAILED {row['filename']}: {e}")

log.info(f"LOCA2 subsets: {done} downloaded, {skipped} already present, {failed} failed")

2026-07-29 16:45:02 | WARNING | 01_extract_loca2 | h5netcdf not installed - skipping subsets. conda install -n arcgispro-py3 -c conda-forge h5netcdf


2026-07-29 16:45:02 | INFO | 01_extract_loca2 | LOCA2 subsets: 0 downloaded, 0 already present, 0 failed
